# Build a reproducible artifact bundle for an air-gapped review

Create a manifest-selected archive, verify every member and digest, and reject traversal, unexpected files, and tampering before a bundle can cross a deployment boundary.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** build deterministic archive bytes from an explicit file allowlist; verify trusted manifest membership, size limits, and hashes before exposing payloads; reject traversal, links, unapproved files, and tampering.

**Prerequisites:** Understand Python byte strings, dictionaries, and file-like objects. This standard-library example stays in memory and requires no container runtime, deployment account, or air-gapped machine.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.



Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Define the reviewed release inputs

All bytes are invented and kept in memory. A stale private-looking file is intentionally outside the manifest to prove that packaging is an allowlist operation, not a recursive directory copy. SHA-256 is an integrity check; it does not authenticate who approved a release.


In [ ]:
import hashlib
import io
import json
import tarfile
from pathlib import PurePosixPath

FILES = {
    "app/config.json": b'{"mode":"demo","max_results":5}\n',
    "app/policy.txt": b"Only reviewed read-only tools may run.\n",
    "models/README.txt": b"Model weights are not bundled in this tutorial.\n",
    "unreviewed-notes.txt": b"Invented private material: never package this file.\n",
}
SELECTED = ["app/config.json", "app/policy.txt", "models/README.txt"]


def safe_name(name):
    path = PurePosixPath(name)
    if (
        not name
        or path.is_absolute()
        or ".." in path.parts
        or "\\" in name
        or "\x00" in name
        or str(path) != name
    ):
        raise ValueError("Unsafe archive member")
    return name


manifest = {
    safe_name(n): {
        "size": len(FILES[n]),
        "sha256": hashlib.sha256(FILES[n]).hexdigest(),
    }
    for n in SELECTED
}
manifest_bytes = json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode()
print(json.dumps(manifest, indent=2))

## 2. Produce deterministic tar bytes

Sorted names and fixed metadata make byte-for-byte reproducibility possible. There are no symlinks, ownership records, shell commands, image pulls, or hidden source-tree files. A real container bundle also needs approved immutable image digests and a software bill of materials.


In [ ]:
def build_bundle(files, selected):
    buffer = io.BytesIO()
    with tarfile.open(fileobj=buffer, mode="w", format=tarfile.USTAR_FORMAT) as archive:
        for name in sorted(selected):
            safe_name(name)
            content = files[name]
            member = tarfile.TarInfo(name)
            member.size = len(content)
            member.mode = 0o600
            member.uid = member.gid = member.mtime = 0
            member.uname = member.gname = ""
            archive.addfile(member, io.BytesIO(content))
    return buffer.getvalue()


bundle = build_bundle(FILES, SELECTED)
assert bundle == build_bundle(FILES, list(reversed(SELECTED)))
assert b"unreviewed-notes" not in bundle
release_digest = hashlib.sha256(bundle).hexdigest()
print(
    {
        "archive_bytes": len(bundle),
        "release_sha256": release_digest,
        "review_manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest(),
    }
)

## 3. Verify before making release bytes available

Verification happens fully before returning any payload. Bound the total size and number of members, reject duplicate names and links, require exact manifest membership, and compare content hashes. The demo returns a file map; production extraction should stage into a fresh private directory and promote only after verification.


In [ ]:
def verify_bundle(payload, approved_manifest, expected_digest, max_bytes=1_000_000):
    if len(payload) > max_bytes:
        raise ValueError("Archive size exceeds policy")
    if hashlib.sha256(payload).hexdigest() != expected_digest:
        raise ValueError("Release checksum mismatch")
    verified = {}
    with tarfile.open(fileobj=io.BytesIO(payload), mode="r:") as archive:
        total = 0
        for index, member in enumerate(archive):
            if index >= 100 or not member.isfile():
                raise ValueError("Unsupported member type or count")
            name = safe_name(member.name)
            if name in verified or name not in approved_manifest:
                raise ValueError("Duplicate or unapproved member")
            expected = approved_manifest[name]
            total += member.size
            if total > max_bytes or member.size != expected["size"]:
                raise ValueError("Unexpected expanded size")
            content = archive.extractfile(member).read(member.size + 1)
            if (
                len(content) != member.size
                or hashlib.sha256(content).hexdigest() != expected["sha256"]
            ):
                raise ValueError("Member checksum mismatch")
            verified[name] = content
    if set(verified) != set(approved_manifest):
        raise ValueError("Missing approved member")
    return verified


restored = verify_bundle(bundle, manifest, release_digest)
assert restored == {n: FILES[n] for n in SELECTED}
print({"verified_members": sorted(restored), "ready_for_manual_release_review": True})

## 4. Exercise rejection paths without extracting unsafe content

A malicious member must be rejected even if its overall archive hash is recomputed. This illustrates why a trusted manifest is necessary in addition to a transport checksum.


In [ ]:
def one_member(name, content=b"unsafe", type_=tarfile.REGTYPE):
    buffer = io.BytesIO()
    with tarfile.open(fileobj=buffer, mode="w") as archive:
        item = tarfile.TarInfo(name)
        item.type = type_
        item.size = len(content) if type_ == tarfile.REGTYPE else 0
        archive.addfile(item, io.BytesIO(content) if item.size else None)
    return buffer.getvalue()


rejected = 0
for candidate in [
    one_member("../escape"),
    one_member("/absolute"),
    one_member("unreviewed-notes.txt"),
    one_member("app/config.json", type_=tarfile.SYMTYPE),
    bundle[:-1] + b"X",
]:
    digest = (
        hashlib.sha256(candidate).hexdigest() if candidate != bundle[:-1] + b"X" else release_digest
    )
    try:
        verify_bundle(candidate, manifest, digest)
    except (ValueError, tarfile.TarError):
        rejected += 1
    else:
        raise AssertionError("Invalid bundle accepted")
assert rejected == 5
print({"rejection_tests": rejected, "network_calls": 0, "deployment_actions": 0})

## Expected local results

The release contains exactly three reviewed files and is reproducible across selection order. All five malicious or altered bundle cases are rejected. The unreviewed notes never appear in the release. No files are extracted to disk.

**Try next:** add a detached-signature verification stage with a separately distributed trusted public key before allowing promotion.


## Interpretation, limitations, and cleanup

This notebook validates a release artifact, not a deployment. A digest transmitted beside an archive can be replaced by the same attacker; deliver the reviewed manifest through an authenticated channel or verify an approved detached signature. Production readiness also requires container CVE audits, signature policy, trusted host keys, destination access controls, backup/rollback, and verification on the isolated target. No cloud account, SSH connection, presigned URL, or container runtime is used here.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [Python tar extraction safety](https://docs.python.org/3/library/tarfile.html#extraction-filters)
- [Python SHA-256 hashing](https://docs.python.org/3/library/hashlib.html)
